# RLHF with Proximal Policy Optimization (PPO)

## Overview

Proximal Policy Optimization (PPO) is the dominant algorithm for RLHF in language models.

### Historical Development
- **2017**: PPO paper (Schulman et al., OpenAI)
- **2019**: Fine-tuning language models from human preferences (Ziegler et al.)
- **2020**: Learning to summarize with human feedback (Stiennon et al.)
- **2022**: InstructGPT uses PPO to align GPT-3 (Ouyang et al.)
- **2023**: ChatGPT scales PPO to production

### Why PPO for RLHF?
1. **Stable**: Clipped objective prevents destructive updates
2. **Sample efficient**: Reuses data with importance sampling
3. **Simple**: Minimal hyperparameters
4. **Proven**: Works at scale (175B parameters)

### RLHF Pipeline with PPO
1. **SFT**: Supervised fine-tuning on demonstrations
2. **Reward Model**: Train on human preferences
3. **PPO**: Optimize policy using reward model + KL penalty

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from typing import List, Dict, Tuple, Optional, NamedTuple
from dataclasses import dataclass
from tqdm import tqdm
import math
from collections import deque

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

# Set random seeds
torch.manual_seed(42)
np.random.seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

## 1. PPO Algorithm Foundation

### Policy Gradient Basics

Goal: Maximize expected reward
$$J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} [R(\tau)]$$

Policy gradient:
$$\nabla_\theta J(\theta) = \mathbb{E}_{\tau \sim \pi_\theta} \left[ \sum_{t=0}^T \nabla_\theta \log \pi_\theta(a_t | s_t) A^\pi(s_t, a_t) \right]$$

where $A^\pi(s, a)$ is the advantage function: how much better is action $a$ than average?

### The Problem with Vanilla PG

Large policy updates can be destructive:
- May collapse performance
- Hard to tune learning rate
- Poor sample efficiency

### PPO Solution: Clipped Objective

Instead of directly maximizing policy gradient, PPO uses:

$$L^{CLIP}(\theta) = \mathbb{E}_t \left[ \min\left( r_t(\theta) A_t, \text{clip}(r_t(\theta), 1-\epsilon, 1+\epsilon) A_t \right) \right]$$

where:
- $r_t(\theta) = \frac{\pi_\theta(a_t | s_t)}{\pi_{\theta_{old}}(a_t | s_t)}$ is the probability ratio
- $\epsilon$ is the clip range (typically 0.2)
- $A_t$ is the advantage estimate

### Key Insight
The clipping prevents large policy updates:
- If $A_t > 0$ (good action), clip prevents excessive increase
- If $A_t < 0$ (bad action), clip prevents excessive decrease
- Keeps policy close to old policy (proximal)

In [ ]:
def visualize_ppo_clipping():
    """Visualize PPO clipping mechanism."""
    
    ratios = np.linspace(0.5, 1.5, 100)
    epsilon = 0.2
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Positive advantage
    A_pos = 1.0
    unclipped_pos = ratios * A_pos
    clipped_pos = np.minimum(unclipped_pos, (1 + epsilon) * A_pos)
    
    ax1.plot(ratios, unclipped_pos, '--', label='Unclipped objective', linewidth=2, alpha=0.7)
    ax1.plot(ratios, clipped_pos, '-', label='PPO clipped objective', linewidth=2)
    ax1.axvline(x=1, color='gray', linestyle=':', alpha=0.5)
    ax1.axvline(x=1+epsilon, color='red', linestyle='--', alpha=0.5, label=f'Clip boundary (1+ε)')
    ax1.set_xlabel('Probability Ratio r(θ)', fontsize=12)
    ax1.set_ylabel('Objective Value', fontsize=12)
    ax1.set_title('PPO Clipping (Positive Advantage)', fontsize=14)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Negative advantage
    A_neg = -1.0
    unclipped_neg = ratios * A_neg
    clipped_neg = np.maximum(unclipped_neg, (1 - epsilon) * A_neg)
    
    ax2.plot(ratios, unclipped_neg, '--', label='Unclipped objective', linewidth=2, alpha=0.7)
    ax2.plot(ratios, clipped_neg, '-', label='PPO clipped objective', linewidth=2)
    ax2.axvline(x=1, color='gray', linestyle=':', alpha=0.5)
    ax2.axvline(x=1-epsilon, color='red', linestyle='--', alpha=0.5, label=f'Clip boundary (1-ε)')
    ax2.set_xlabel('Probability Ratio r(θ)', fontsize=12)
    ax2.set_ylabel('Objective Value', fontsize=12)
    ax2.set_title('PPO Clipping (Negative Advantage)', fontsize=14)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== PPO Clipping Mechanism ===")
    print("\nPositive advantage (good action):")
    print("  - Encourages higher probability")
    print("  - But clipped to prevent excessive increase")
    print("\nNegative advantage (bad action):")
    print("  - Discourages action (lower probability)")
    print("  - But clipped to prevent excessive decrease")
    print("\nResult: Conservative, stable updates")

visualize_ppo_clipping()

## 2. Actor-Critic Architecture

### Components

1. **Actor (Policy)**: $\pi_\theta(a|s)$ - generates actions
2. **Critic (Value)**: $V_\phi(s)$ - estimates state value

### Why Critic?

The critic provides advantage estimates:
$$A(s_t, a_t) = Q(s_t, a_t) - V(s_t)$$

Using Generalized Advantage Estimation (GAE):
$$A_t^{GAE} = \sum_{l=0}^\infty (\gamma \lambda)^l \delta_{t+l}$$

where $\delta_t = r_t + \gamma V(s_{t+1}) - V(s_t)$ is the TD error.

### For Language Models

```
Actor (Policy):  [prompt] → [completion] with probabilities
Critic (Value):  [prompt] [partial completion] → value estimate
```

Often share parameters:
```
Input → Transformer → {Policy Head, Value Head}
```

In [ ]:
class ActorCriticModel(nn.Module):
    """Actor-Critic model for language generation."""
    
    def __init__(
        self,
        vocab_size: int,
        hidden_dim: int = 512,
        num_layers: int = 6,
        num_heads: int = 8,
        max_seq_len: int = 512,
        dropout: float = 0.1
    ):
        super().__init__()
        self.hidden_dim = hidden_dim
        
        # Shared transformer backbone
        self.embedding = nn.Embedding(vocab_size, hidden_dim)
        self.pos_embedding = nn.Embedding(max_seq_len, hidden_dim)
        
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_dim,
            nhead=num_heads,
            dim_feedforward=4 * hidden_dim,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers)
        
        # Policy head (actor)
        self.policy_head = nn.Linear(hidden_dim, vocab_size)
        
        # Value head (critic)
        self.value_head = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1)
        )
        
        self.dropout = nn.Dropout(dropout)
        
    def forward(
        self,
        input_ids: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Args:
            input_ids: (batch_size, seq_len)
            attention_mask: (batch_size, seq_len)
        
        Returns:
            logits: (batch_size, seq_len, vocab_size) policy logits
            values: (batch_size, seq_len) value estimates
        """
        batch_size, seq_len = input_ids.shape
        
        # Embeddings
        positions = torch.arange(seq_len, device=input_ids.device).unsqueeze(0)
        x = self.embedding(input_ids) + self.pos_embedding(positions)
        x = self.dropout(x)
        
        # Transformer
        if attention_mask is not None:
            attention_mask = attention_mask.bool()
        
        x = self.transformer(x, src_key_padding_mask=~attention_mask if attention_mask is not None else None)
        
        # Policy logits
        logits = self.policy_head(x)
        
        # Value estimates
        values = self.value_head(x).squeeze(-1)
        
        return logits, values
    
    def get_action_log_probs(
        self,
        input_ids: torch.Tensor,
        actions: torch.Tensor,
        attention_mask: Optional[torch.Tensor] = None
    ) -> torch.Tensor:
        """Get log probabilities for taken actions."""
        logits, _ = self.forward(input_ids, attention_mask)
        log_probs = F.log_softmax(logits, dim=-1)
        
        # Gather log probs for actual actions
        action_log_probs = log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)
        
        return action_log_probs


# Test model
model = ActorCriticModel(vocab_size=1000, hidden_dim=256, num_layers=4)
print(f"Actor-Critic Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

# Test forward pass
batch_size, seq_len = 4, 32
input_ids = torch.randint(0, 1000, (batch_size, seq_len))
attention_mask = torch.ones(batch_size, seq_len)

logits, values = model(input_ids, attention_mask)
print(f"\nLogits shape: {logits.shape}")
print(f"Values shape: {values.shape}")
print(f"Sample values: {values[0, :5]}")

## 3. KL Divergence Constraint

### The Problem

Without constraints, RL can optimize the reward model too aggressively:
- Reward hacking
- Mode collapse
- Nonsensical outputs

### Solution: KL Penalty

Add a penalty to keep policy close to reference model:

$$r_{final}(x, y) = r_{RM}(x, y) - \beta \cdot D_{KL}(\pi_\theta(\cdot|x) || \pi_{ref}(\cdot|x))$$

where:
- $r_{RM}(x, y)$ is the reward model score
- $\pi_{ref}$ is the reference policy (typically the SFT model)
- $\beta$ controls the strength of the constraint

### Token-Level KL

For language models, KL is computed per token:

$$D_{KL} = \sum_t \left[ \log \frac{\pi_\theta(y_t | x, y_{<t})}{\pi_{ref}(y_t | x, y_{<t})} \right]$$

### Effect of $\beta$
- **High $\beta$**: Conservative, stays close to reference
- **Low $\beta$**: Aggressive, more reward optimization
- **Typical values**: 0.01 - 0.1

In [ ]:
def compute_kl_divergence(
    policy_logits: torch.Tensor,
    ref_logits: torch.Tensor,
    actions: torch.Tensor,
    attention_mask: Optional[torch.Tensor] = None
) -> torch.Tensor:
    """
    Compute KL divergence between policy and reference.
    
    Args:
        policy_logits: (batch_size, seq_len, vocab_size)
        ref_logits: (batch_size, seq_len, vocab_size)
        actions: (batch_size, seq_len) actual tokens
        attention_mask: (batch_size, seq_len)
    
    Returns:
        kl: (batch_size,) KL divergence per sequence
    """
    # Log probabilities
    policy_log_probs = F.log_softmax(policy_logits, dim=-1)
    ref_log_probs = F.log_softmax(ref_logits, dim=-1)
    
    # Get log probs for actual actions
    policy_action_log_probs = policy_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)
    ref_action_log_probs = ref_log_probs.gather(-1, actions.unsqueeze(-1)).squeeze(-1)
    
    # KL = log(policy) - log(ref)
    kl_per_token = policy_action_log_probs - ref_action_log_probs
    
    # Mask and sum
    if attention_mask is not None:
        kl_per_token = kl_per_token * attention_mask
        kl = kl_per_token.sum(dim=1) / attention_mask.sum(dim=1)
    else:
        kl = kl_per_token.mean(dim=1)
    
    return kl


# Visualize effect of KL penalty
def visualize_kl_penalty():
    """Show how KL penalty affects final reward."""
    
    reward_model_scores = np.linspace(0, 10, 100)
    kl_divergences = np.array([0, 1, 2, 5])
    betas = [0.01, 0.05, 0.1]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Fixed KL, varying beta
    kl_fixed = 2.0
    for beta in betas:
        final_rewards = reward_model_scores - beta * kl_fixed
        axes[0].plot(reward_model_scores, final_rewards, 
                     label=f'β={beta}', linewidth=2)
    
    axes[0].set_xlabel('Reward Model Score', fontsize=12)
    axes[0].set_ylabel('Final Reward (with KL penalty)', fontsize=12)
    axes[0].set_title(f'Effect of β (KL={kl_fixed})', fontsize=14)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Fixed beta, varying KL
    beta_fixed = 0.05
    rm_score = 8.0
    kl_range = np.linspace(0, 10, 100)
    final_rewards = rm_score - beta_fixed * kl_range
    
    axes[1].plot(kl_range, final_rewards, linewidth=2)
    axes[1].axhline(y=rm_score, color='r', linestyle='--', 
                     alpha=0.5, label=f'No penalty (RM score={rm_score})')
    axes[1].set_xlabel('KL Divergence', fontsize=12)
    axes[1].set_ylabel('Final Reward', fontsize=12)
    axes[1].set_title(f'Penalty vs KL (β={beta_fixed})', fontsize=14)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== KL Penalty Effects ===")
    print("\nHigher β → stronger constraint → stays closer to reference")
    print("Higher KL → larger penalty → lower final reward")
    print("\nTrade-off: reward optimization vs distribution shift")

visualize_kl_penalty()

## 4. Full RLHF Pipeline

### Step-by-Step Process

**Phase 1: Supervised Fine-Tuning**
1. Train base model on demonstrations
2. Creates initial policy $\pi_{SFT}$

**Phase 2: Reward Modeling**
1. Collect preference comparisons
2. Train reward model $r_\theta(x, y)$

**Phase 3: PPO Training Loop**
```
Initialize policy π_θ from π_SFT
Reference policy π_ref ← π_SFT (frozen)

For each iteration:
    1. Rollout: Generate completions with π_θ
    2. Score: Compute rewards with r_θ
    3. KL penalty: r_final = r_θ - β·KL(π_θ || π_ref)
    4. GAE: Compute advantages
    5. PPO: Update policy with clipped objective
    6. Value update: Train critic
```

### Hyperparameters
- **PPO epochs**: 4-8 per batch
- **Batch size**: 128-512 sequences
- **Clip range $\epsilon$**: 0.2
- **KL coefficient $\beta$**: 0.02-0.1
- **Learning rate**: 1e-6 to 1e-5
- **GAE $\lambda$**: 0.95

In [ ]:
class PPOTrainer:
    """PPO trainer for RLHF."""
    
    def __init__(
        self,
        policy_model: nn.Module,
        ref_model: nn.Module,
        reward_model: nn.Module,
        clip_range: float = 0.2,
        kl_coef: float = 0.05,
        value_coef: float = 0.5,
        entropy_coef: float = 0.01,
        gamma: float = 0.99,
        gae_lambda: float = 0.95,
        ppo_epochs: int = 4,
        learning_rate: float = 1e-5
    ):
        self.policy_model = policy_model
        self.ref_model = ref_model
        self.reward_model = reward_model
        
        # Hyperparameters
        self.clip_range = clip_range
        self.kl_coef = kl_coef
        self.value_coef = value_coef
        self.entropy_coef = entropy_coef
        self.gamma = gamma
        self.gae_lambda = gae_lambda
        self.ppo_epochs = ppo_epochs
        
        # Optimizer
        self.optimizer = torch.optim.AdamW(
            policy_model.parameters(),
            lr=learning_rate
        )
        
        # Freeze reference and reward models
        for param in self.ref_model.parameters():
            param.requires_grad = False
        for param in self.reward_model.parameters():
            param.requires_grad = False
    
    def compute_gae(
        self,
        rewards: torch.Tensor,
        values: torch.Tensor,
        masks: torch.Tensor
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        """
        Compute Generalized Advantage Estimation.
        
        Args:
            rewards: (batch_size, seq_len)
            values: (batch_size, seq_len)
            masks: (batch_size, seq_len)
        
        Returns:
            advantages: (batch_size, seq_len)
            returns: (batch_size, seq_len)
        """
        batch_size, seq_len = rewards.shape
        advantages = torch.zeros_like(rewards)
        lastgaelam = 0
        
        for t in reversed(range(seq_len)):
            if t == seq_len - 1:
                nextnonterminal = 0
                nextvalues = 0
            else:
                nextnonterminal = masks[:, t + 1]
                nextvalues = values[:, t + 1]
            
            delta = rewards[:, t] + self.gamma * nextvalues * nextnonterminal - values[:, t]
            advantages[:, t] = lastgaelam = delta + self.gamma * self.gae_lambda * nextnonterminal * lastgaelam
        
        returns = advantages + values
        return advantages, returns
    
    def ppo_loss(
        self,
        log_probs: torch.Tensor,
        old_log_probs: torch.Tensor,
        advantages: torch.Tensor,
        values: torch.Tensor,
        returns: torch.Tensor,
        entropy: torch.Tensor,
        mask: torch.Tensor
    ) -> Dict[str, torch.Tensor]:
        """
        Compute PPO loss.
        
        Returns:
            Dictionary with loss components
        """
        # Normalize advantages
        advantages = (advantages - advantages.mean()) / (advantages.std() + 1e-8)
        
        # Policy loss (clipped)
        ratio = torch.exp(log_probs - old_log_probs)
        pg_loss1 = -advantages * ratio
        pg_loss2 = -advantages * torch.clamp(ratio, 1 - self.clip_range, 1 + self.clip_range)
        pg_loss = torch.max(pg_loss1, pg_loss2)
        pg_loss = (pg_loss * mask).sum() / mask.sum()
        
        # Value loss
        value_loss = F.mse_loss(values, returns, reduction='none')
        value_loss = (value_loss * mask).sum() / mask.sum()
        
        # Entropy bonus (encourages exploration)
        entropy_loss = -(entropy * mask).sum() / mask.sum()
        
        # Total loss
        total_loss = pg_loss + self.value_coef * value_loss + self.entropy_coef * entropy_loss
        
        return {
            'total_loss': total_loss,
            'policy_loss': pg_loss,
            'value_loss': value_loss,
            'entropy_loss': entropy_loss,
            'approx_kl': ((ratio - 1) - torch.log(ratio)).mean()
        }


print("PPO Trainer initialized with:")
print("- Clipped policy objective")
print("- GAE for advantage estimation")
print("- Value function loss")
print("- Entropy bonus for exploration")
print("- KL penalty to reference model")

## 5. Using TRL Library

### HuggingFace TRL

The Transformer Reinforcement Learning (TRL) library provides production-ready RLHF:

```python
from trl import PPOTrainer, PPOConfig, AutoModelForCausalLMWithValueHead

# Load models
model = AutoModelForCausalLMWithValueHead.from_pretrained('gpt2')
ref_model = AutoModelForCausalLMWithValueHead.from_pretrained('gpt2')
reward_model = pipeline('sentiment-analysis', 'lvwerra/distilbert-imdb')

# Configure PPO
config = PPOConfig(
    learning_rate=1e-5,
    batch_size=128,
    mini_batch_size=32,
    ppo_epochs=4,
    init_kl_coef=0.05,
    cliprange=0.2,
    vf_coef=0.5,
)

# Initialize trainer
ppo_trainer = PPOTrainer(
    config=config,
    model=model,
    ref_model=ref_model,
    tokenizer=tokenizer,
)

# Training loop
for batch in dataloader:
    # Generate completions
    query_tensors = batch['input_ids']
    response_tensors = ppo_trainer.generate(query_tensors)
    
    # Compute rewards
    texts = [tokenizer.decode(r) for r in response_tensors]
    rewards = [reward_model(text)[0]['score'] for text in texts]
    
    # PPO update
    stats = ppo_trainer.step(query_tensors, response_tensors, rewards)
```

### Key Features
1. **Automatic batching**: Handles memory efficiently
2. **KL penalties**: Built-in reference model comparison
3. **Adaptive KL**: Adjusts coefficient dynamically
4. **Logging**: Tracks all metrics
5. **Distributed**: Multi-GPU support

In [ ]:
# Simulate TRL-style training loop
def simulate_ppo_training(
    num_iterations: int = 10,
    baseline_reward: float = 5.0
) -> Dict[str, List[float]]:
    """Simulate PPO training dynamics."""
    
    history = {
        'iteration': [],
        'reward_mean': [],
        'reward_std': [],
        'kl_divergence': [],
        'policy_loss': [],
        'value_loss': [],
    }
    
    kl_coef = 0.05
    
    for i in range(num_iterations):
        # Simulate improving rewards but increasing KL
        reward_improvement = i * 0.3
        kl_increase = i * 0.2
        
        # Add noise
        reward_mean = baseline_reward + reward_improvement + np.random.normal(0, 0.2)
        reward_std = 1.0 + np.random.normal(0, 0.1)
        kl_div = kl_increase + np.random.normal(0, 0.1)
        
        # Simulate losses decreasing
        policy_loss = 0.5 * np.exp(-i * 0.2) + np.random.normal(0, 0.05)
        value_loss = 0.3 * np.exp(-i * 0.15) + np.random.normal(0, 0.03)
        
        history['iteration'].append(i)
        history['reward_mean'].append(reward_mean)
        history['reward_std'].append(reward_std)
        history['kl_divergence'].append(kl_div)
        history['policy_loss'].append(policy_loss)
        history['value_loss'].append(value_loss)
    
    return history


def plot_ppo_training(history: Dict[str, List[float]]):
    """Plot PPO training curves."""
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    iterations = history['iteration']
    
    # Rewards
    axes[0, 0].plot(iterations, history['reward_mean'], 'o-', linewidth=2)
    axes[0, 0].fill_between(
        iterations,
        np.array(history['reward_mean']) - np.array(history['reward_std']),
        np.array(history['reward_mean']) + np.array(history['reward_std']),
        alpha=0.3
    )
    axes[0, 0].set_xlabel('Iteration', fontsize=12)
    axes[0, 0].set_ylabel('Reward', fontsize=12)
    axes[0, 0].set_title('Reward Progress', fontsize=14)
    axes[0, 0].grid(True, alpha=0.3)
    
    # KL divergence
    axes[0, 1].plot(iterations, history['kl_divergence'], 'o-', 
                    color='orange', linewidth=2)
    axes[0, 1].set_xlabel('Iteration', fontsize=12)
    axes[0, 1].set_ylabel('KL Divergence', fontsize=12)
    axes[0, 1].set_title('Distribution Shift', fontsize=14)
    axes[0, 1].grid(True, alpha=0.3)
    
    # Policy loss
    axes[1, 0].plot(iterations, history['policy_loss'], 'o-', 
                    color='green', linewidth=2)
    axes[1, 0].set_xlabel('Iteration', fontsize=12)
    axes[1, 0].set_ylabel('Policy Loss', fontsize=12)
    axes[1, 0].set_title('Policy Optimization', fontsize=14)
    axes[1, 0].grid(True, alpha=0.3)
    
    # Value loss
    axes[1, 1].plot(iterations, history['value_loss'], 'o-', 
                    color='red', linewidth=2)
    axes[1, 1].set_xlabel('Iteration', fontsize=12)
    axes[1, 1].set_ylabel('Value Loss', fontsize=12)
    axes[1, 1].set_title('Critic Training', fontsize=14)
    axes[1, 1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()


# Simulate and plot
print("Simulating PPO training...\n")
history = simulate_ppo_training(num_iterations=20)
plot_ppo_training(history)

print("\nTypical training dynamics:")
print("1. Rewards increase as policy improves")
print("2. KL divergence grows (policy drifts from reference)")
print("3. Policy loss decreases (better optimization)")
print("4. Value loss decreases (critic learns to predict)")

## 6. Reward Shaping

### The Challenge

Raw reward model scores may not be ideal for RL:
- Sparse rewards (only at sequence end)
- Scale mismatch
- Optimization difficulties

### Techniques

**1. Reward Scaling**
$$r_{scaled} = \frac{r - \mu}{\sigma}$$
Normalize to zero mean, unit variance

**2. Reward Clipping**
$$r_{clipped} = \text{clip}(r, r_{min}, r_{max})$$
Prevent outliers from dominating

**3. Dense Rewards**
Provide intermediate rewards during generation:
- Length penalties
- Perplexity bonuses
- Coherence scores

**4. Reward Whitening**
Per-batch normalization:
$$r_{whitened} = \frac{r - \text{mean}_{batch}(r)}{\text{std}_{batch}(r)}$$

**5. Advantage Centering**
Zero-center advantages:
$$A_{centered} = A - \text{mean}(A)$$

In [ ]:
def visualize_reward_shaping():
    """Show effects of different reward shaping techniques."""
    
    # Generate skewed reward distribution
    np.random.seed(42)
    raw_rewards = np.concatenate([
        np.random.normal(5, 2, 80),
        np.random.normal(15, 1, 20)  # Some outliers
    ])
    
    # Apply different shaping techniques
    rewards_scaled = (raw_rewards - raw_rewards.mean()) / raw_rewards.std()
    rewards_clipped = np.clip(raw_rewards, 0, 10)
    rewards_whitened = (raw_rewards - raw_rewards.mean()) / (raw_rewards.std() + 1e-8)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Raw rewards
    axes[0, 0].hist(raw_rewards, bins=30, color='blue', alpha=0.7)
    axes[0, 0].axvline(raw_rewards.mean(), color='r', linestyle='--', 
                       linewidth=2, label=f'Mean={raw_rewards.mean():.2f}')
    axes[0, 0].set_xlabel('Reward', fontsize=12)
    axes[0, 0].set_ylabel('Count', fontsize=12)
    axes[0, 0].set_title('Raw Rewards', fontsize=14)
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)
    
    # Scaled rewards
    axes[0, 1].hist(rewards_scaled, bins=30, color='green', alpha=0.7)
    axes[0, 1].axvline(0, color='r', linestyle='--', linewidth=2, label='Mean=0')
    axes[0, 1].set_xlabel('Reward', fontsize=12)
    axes[0, 1].set_ylabel('Count', fontsize=12)
    axes[0, 1].set_title('Scaled Rewards (normalized)', fontsize=14)
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)
    
    # Clipped rewards
    axes[1, 0].hist(rewards_clipped, bins=30, color='orange', alpha=0.7)
    axes[1, 0].axvline(10, color='r', linestyle='--', linewidth=2, label='Clip max=10')
    axes[1, 0].set_xlabel('Reward', fontsize=12)
    axes[1, 0].set_ylabel('Count', fontsize=12)
    axes[1, 0].set_title('Clipped Rewards', fontsize=14)
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)
    
    # Comparison statistics
    stats_text = f"""
    Raw Rewards:
      Mean: {raw_rewards.mean():.2f}
      Std:  {raw_rewards.std():.2f}
      Min:  {raw_rewards.min():.2f}
      Max:  {raw_rewards.max():.2f}
    
    Scaled:
      Mean: {rewards_scaled.mean():.2f}
      Std:  {rewards_scaled.std():.2f}
    
    Clipped:
      Mean: {rewards_clipped.mean():.2f}
      Max:  {rewards_clipped.max():.2f}
    """
    axes[1, 1].text(0.1, 0.5, stats_text, fontsize=11, family='monospace',
                    verticalalignment='center')
    axes[1, 1].axis('off')
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Reward Shaping Benefits ===")
    print("\n1. Scaling: Normalizes to zero mean, unit variance")
    print("   - Better gradient flow")
    print("   - Comparable across batches")
    print("\n2. Clipping: Removes outliers")
    print("   - Prevents single examples from dominating")
    print("   - More stable training")

visualize_reward_shaping()

## 7. Stability Issues and Solutions

### Common Problems

**1. Reward Hacking**
- **Symptom**: High rewards but nonsensical outputs
- **Solution**: Stronger KL penalty, ensemble reward models

**2. Training Collapse**
- **Symptom**: Policy produces repetitive or degenerate text
- **Solution**: Early stopping, KL constraints, entropy bonus

**3. Exploding KL**
- **Symptom**: KL divergence grows rapidly
- **Solution**: Adaptive KL coefficient, smaller learning rate

**4. Value Function Mismatch**
- **Symptom**: Poor advantage estimates, unstable training
- **Solution**: More value function updates, separate learning rate

**5. Distribution Shift**
- **Symptom**: Reward model fails on policy outputs
- **Solution**: Iterative data collection, uncertainty penalties

### Best Practices

1. **Gradual optimization**: Small learning rates, conservative updates
2. **Monitoring**: Track KL, entropy, reward variance
3. **Regularization**: KL penalty, entropy bonus, gradient clipping
4. **Checkpointing**: Save frequently, easy rollback
5. **Human evaluation**: Regular quality checks

In [ ]:
def simulate_training_issues():
    """Simulate common training issues and their fixes."""
    
    iterations = np.arange(0, 100)
    
    # Scenario 1: Unstable training (no clipping)
    reward_unstable = 5 + np.cumsum(np.random.randn(100) * 2)
    kl_unstable = np.abs(np.cumsum(np.random.randn(100) * 0.5))
    
    # Scenario 2: Stable training (with PPO)
    reward_stable = 5 + 0.1 * iterations + np.random.randn(100) * 0.3
    kl_stable = 0.5 + 0.01 * iterations + np.random.randn(100) * 0.05
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Rewards comparison
    axes[0].plot(iterations, reward_unstable, label='Without PPO', 
                 alpha=0.7, linewidth=2)
    axes[0].plot(iterations, reward_stable, label='With PPO', 
                 alpha=0.7, linewidth=2)
    axes[0].set_xlabel('Iteration', fontsize=12)
    axes[0].set_ylabel('Reward', fontsize=12)
    axes[0].set_title('Training Stability: Rewards', fontsize=14)
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # KL divergence comparison
    axes[1].plot(iterations, kl_unstable, label='Without constraints',
                 alpha=0.7, linewidth=2)
    axes[1].plot(iterations, kl_stable, label='With KL penalty',
                 alpha=0.7, linewidth=2)
    axes[1].set_xlabel('Iteration', fontsize=12)
    axes[1].set_ylabel('KL Divergence', fontsize=12)
    axes[1].set_title('Training Stability: Distribution Shift', fontsize=14)
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Stability Solutions ===")
    print("\n1. PPO clipping → stable reward progress")
    print("2. KL penalty → controlled distribution shift")
    print("3. Gradient clipping → prevents explosions")
    print("4. Entropy bonus → maintains diversity")
    print("5. Early stopping → prevents overfitting")

simulate_training_issues()

## 8. Evaluation of Aligned Models

### Evaluation Dimensions

**1. Reward Model Score**
- Check average reward improves
- But beware of overfitting to RM

**2. Human Evaluation**
- A/B testing vs baseline
- Multi-aspect ratings (helpful, harmless, honest)
- Win rate analysis

**3. Automatic Metrics**
- Perplexity (fluency)
- BLEU/ROUGE (when references available)
- BERTScore (semantic similarity)

**4. Safety Evaluation**
- Red teaming
- Adversarial prompts
- Refusal rate on unsafe requests

**5. Capability Benchmarks**
- MMLU, HellaSwag, etc.
- Check for alignment tax
- Task-specific performance

### Diagnostic Checks

```python
# 1. KL divergence vs baseline
kl = compute_kl(policy_outputs, reference_outputs)
assert kl < kl_threshold  # Not too far from reference

# 2. Entropy check
entropy = compute_entropy(policy_outputs)
assert entropy > min_entropy  # Not collapsed

# 3. Reward distribution
check_reward_stats(rewards)

# 4. Output diversity
unique_outputs = len(set(outputs)) / len(outputs)
assert unique_outputs > diversity_threshold
```

In [ ]:
def evaluate_alignment_quality(
    baseline_score: float = 5.0,
    aligned_score: float = 7.5,
    kl_divergence: float = 2.0
):
    """Visualize alignment quality metrics."""
    
    metrics = ['Helpfulness', 'Harmlessness', 'Honesty', 'Coherence', 'Overall']
    baseline_scores = [5.0, 6.0, 5.5, 6.5, 5.5]
    aligned_scores = [7.5, 8.0, 7.0, 7.8, 7.8]
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Comparison bar chart
    x = np.arange(len(metrics))
    width = 0.35
    
    bars1 = axes[0].bar(x - width/2, baseline_scores, width, 
                         label='Baseline (SFT)', alpha=0.8)
    bars2 = axes[0].bar(x + width/2, aligned_scores, width,
                         label='RLHF Aligned', alpha=0.8)
    
    axes[0].set_ylabel('Score (0-10)', fontsize=12)
    axes[0].set_title('Human Evaluation Comparison', fontsize=14)
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(metrics, rotation=45, ha='right')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3, axis='y')
    axes[0].set_ylim(0, 10)
    
    # Trade-offs visualization
    improvements = np.array(aligned_scores) - np.array(baseline_scores)
    
    data = {
        'Metric': ['Reward', 'KL', 'Improvement', 'Alignment Tax'],
        'Value': [
            (aligned_score - baseline_score) / baseline_score * 100,
            kl_divergence,
            np.mean(improvements),
            -0.5  # Simulated capability loss
        ]
    }
    
    colors = ['green', 'orange', 'blue', 'red']
    bars = axes[1].barh(data['Metric'], data['Value'], color=colors, alpha=0.7)
    axes[1].set_xlabel('Value', fontsize=12)
    axes[1].set_title('Alignment Trade-offs', fontsize=14)
    axes[1].axvline(x=0, color='black', linestyle='-', linewidth=0.8)
    axes[1].grid(True, alpha=0.3, axis='x')
    
    plt.tight_layout()
    plt.show()
    
    print("\n=== Evaluation Summary ===")
    print(f"\nReward improvement: {aligned_score - baseline_score:.1f} points")
    print(f"KL divergence: {kl_divergence:.2f}")
    print(f"Average metric improvement: {np.mean(improvements):.1f}")
    print("\nKey findings:")
    print("+ Significant improvements in helpfulness and harmlessness")
    print("+ Maintained coherence and fluency")
    print("- Small capability loss (alignment tax)")
    print("- Moderate distribution shift from reference")

evaluate_alignment_quality()

## Summary

### Key Takeaways

1. **PPO** is the dominant algorithm for RLHF due to stability and simplicity

2. **Clipped objective** prevents destructive policy updates

3. **Actor-Critic** architecture combines policy and value functions

4. **KL divergence penalty** prevents reward hacking and mode collapse

5. **TRL library** provides production-ready RLHF implementation

6. **Reward shaping** is critical for stable training

7. **Stability requires** careful hyperparameters and monitoring

8. **Evaluation** should be multi-faceted: rewards, human ratings, safety

### Limitations of PPO/RLHF

1. **Computational cost**: Requires multiple models (policy, reference, reward, critic)
2. **Sample inefficiency**: Many rollouts needed
3. **Hyperparameter sensitive**: Requires careful tuning
4. **Reward model dependence**: Quality limited by RM
5. **Training instability**: Can collapse or diverge

### Next: Simpler Alternatives

- **Notebook 42**: DPO (Direct Preference Optimization)
- **Notebook 43**: Constitutional AI
- **Notebook 44**: Domain Adaptation

### References

1. Schulman et al. (2017): "Proximal Policy Optimization Algorithms"
2. Ziegler et al. (2019): "Fine-Tuning Language Models from Human Preferences"
3. Stiennon et al. (2020): "Learning to Summarize with Human Feedback"
4. Ouyang et al. (2022): "Training language models to follow instructions with human feedback"
5. Schulman et al. (2016): "High-Dimensional Continuous Control Using Generalized Advantage Estimation"
6. von Werra et al. (2020): "TRL: Transformer Reinforcement Learning"